In [ ]:
# model path
model_path = "D:/tierra/outputs/plots/Mexico-orgc-tha/Random Forest.joblib"
# data path
new_data_path = "D:/tierra/outputs/plots/Mexico-orgc-tha/preprocessed_data_orgc.csv"
# output path
output_path = "D:/tierra/outputs/plots/Mexico-orgc-tha/predictions.csv"

regressor_name = "Randmom Forest"
TARGET = 'orgc'
UNITS = 't/ha'
COUNTRY = "Mexico"

In [ ]:
import os
import joblib
import pandas as pd

# Load the pipeline
trained_model = joblib.load(model_path)

# load the new data
new_data = pd.read_csv(new_data_path)

# preprocess the new data
def preprocess_data(data):
    # remove missing values
    data = data.dropna()
    # remove duplicates
    data = data.drop_duplicates()
    
    return data

# Predict on new data
df = preprocess_data(new_data)
# Make predictions
predictions = trained_model.predict(df)
# make the directory if it does not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)
# append predicted column to the dataframe
predictions_df = pd.DataFrame(df)
target_column = 'predicted_' + TARGET
predictions_df[target_column] = predictions
# save the predictions to a csv file
predictions_df.to_csv(output_path, index=False)

predictions_df.head()

In [ ]:
# %pip install folium

In [ ]:
import matplotlib.pyplot as plt
import folium # type:ignore
from IPython.display import display

def plot_predictions_on_map(df, lat_col="latitude", lon_col="longitude", value_col="Predicted"):
    # Center map
    center = [df[lat_col].mean(), df[lon_col].mean()]
    m = folium.Map(location=center, zoom_start=6)

    for _, row in df.iterrows():
        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=5,
            popup=f"{value_col}: {row[value_col]:.2f} {UNITS}",
            color='blue',
            fill=True,
            fill_opacity=0.7
        ).add_to(m)
    return m

# Call the function if lat/lon are available
if {'latitude', 'longitude'}.issubset(predictions_df.columns):
    m = plot_predictions_on_map(predictions_df, value_col=target_column)
    display(m)
else:
    print("Latitude and Longitude columns not found. Skipping map.")

In [ ]:
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def plot_regression_line(df, true_col, pred_col):
    plt.figure(figsize=(8, 6))
    sns.regplot(x=true_col, y=pred_col, data=df, line_kws={"color": "red", "alpha": 0.7})
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Regression Line: Predicted vs Actual")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Only if you have actual target values
if TARGET in predictions_df.columns:
    plot_regression_line(predictions_df, TARGET, target_column)
    # print metrics
    y_true = predictions_df[TARGET]
    y_pred = predictions_df[target_column]
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print("-" * 30)
    print(f"MAE: {mae:.2f} {UNITS}")
    print(f"MSE: {mse:.2f} {UNITS}^2")
    print(f"RMSE: {np.sqrt(mse):.2f} {UNITS}")
    print(f"R^2: {r2:.2f}")
    print("-" * 30)
    
else:
    print("No actual target values found. Skipping regression plot.")

In [ ]:
def plot_prediction_distribution(df, pred_col):
    plt.figure(figsize=(8, 6))
    sns.histplot(df[pred_col], kde=True, color='skyblue')
    plt.title("Distribution of Predicted Values")
    plt.xlabel("Predicted")
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_prediction_distribution(predictions_df, target_column)